# SNC — Train the Query-Conditioned Separator

Trains the **scalable** separation model: a U-Net that is told *which*
class to extract and emits a single mask for it. It handles all
ESC-50 + UrbanSound8K classes with one architecture.

Mixtures are generated **on the fly** during training — there is no
multi-gigabyte dataset file.

## Pipeline

| Stage | Script | Output |
|------|--------|--------|
| 0 | (cell) | Download ESC-50 + UrbanSound8K to Drive if missing |
| 1 | `train_conditioned_separator.py` | `separator_unet_film_multi_<version>.h5` + `_classes.json` |
| 2 | `evaluate_conditioned_separator.py` | SI-SDR results table |
| 3 | `colab_webapp.ipynb` | Launch the Gradio app to remove sounds |

## Before running — pick a GPU

**Runtime → Change runtime type → GPU.**

- **L4 or A100** (Colab Pro / Pro+): recommended — 3-5× faster than T4.
  TF 2.20 (installed below) ships kernels for both, so the old
  `CUDA_ERROR_INVALID_PTX` error no longer applies to them.
- **T4**: works fine, just slower.
- **Avoid the very newest Blackwell cards** — TF 2.20 may still lack PTX
  for them.

## Speed knobs (training)

The training script reads these env vars — safe defaults, no edit needed:

| Env var | Default | Effect |
|---|---|---|
| `SNC_MIXED_PRECISION` | `1` (on) | float16 compute, 1.5-2× faster on Tensor Cores. Set `0` if you ever see NaN loss. |
| `SNC_BATCH_SIZE` | `32` | Bigger batch keeps a fast GPU busy. Push `64` on A100/L4. |
| `SNC_XLA` | `1` (on) | XLA fuses the train step. Set `0` if an op fails to compile. |

These change **speed only** — output layers and the loss are pinned to
float32, so the trained weights are unaffected.

## 1. Mount Drive and clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/snc'
GITHUB_USER = 'keremtutumlu'
GITHUB_REPO_NAME = 'selective-noise-cancelling'
BRANCH = 'feature/v3.0'

# Private repo? Add a Colab Secret named GITHUB_TOKEN (key icon, scope: repo).
import os, subprocess, shutil
from pathlib import Path

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if token:
    clone_url = f'https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git'
    print('Using GITHUB_TOKEN from Colab Secrets.')
else:
    clone_url = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git'
    print('No GITHUB_TOKEN secret found — attempting anonymous clone.')

REPO_DIR = Path(f'/content/{GITHUB_REPO_NAME}')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

# GIT_TERMINAL_PROMPT=0 prevents git from opening an interactive credential
# prompt when there is no TTY (background Colab execution).
result = subprocess.run(
    ['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)],
    capture_output=True, text=True,
    stdin=subprocess.DEVNULL,
    env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'},
)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(
        f'git clone failed (exit {result.returncode}). '
        'If the repo is private, add a GITHUB_TOKEN secret in the left sidebar.'
    )
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-5'], check=True)

In [ ]:
import subprocess
result = subprocess.run(['git', 'stash'], capture_output=True, text=True)
print(result.stdout)
subprocess.run(['git', 'pull', '--rebase', 'origin', 'feature/v3.0'], check=True)
subprocess.run(['git', 'stash', 'pop'], capture_output=True)
subprocess.run(['git', 'log', '--oneline', '-3'], check=True)

In [ ]:
# === Model version — set once, applies to every stage below ===
# Training, smoke test, SI-SDR and detection all read SNC_MODEL_VERSION,
# so bumping this (e.g. 'v2.8') retargets the whole notebook with no
# code edits and no commits. The .h5 and _classes.json checkpoints are
# named after this string.
import os, subprocess, sys
os.environ['SNC_MODEL_VERSION'] = 'v3.0'
# Decoded-dataset cache lives on local SSD, not Drive: the first decode
# of ~10k WAVs from Drive can take over an hour, but the pickle is read
# in seconds on every later script.
os.environ['SNC_DATA_CACHE_DIR'] = '/content'

# === Drive audit log ===
# Every stage appends a structured entry to run_log.json (pretty JSON
# array with version, metrics, last 4 kB of stdout), AND every cell that
# uses scripts/run_logged.py also writes a *full* per-run log file under
# $SNC_DRIVE_LOG_DIR/runs/ — so a disconnected Colab frontend can never
# lose the output. Leave this pointing at Drive so logs persist across
# sessions.
os.environ['SNC_DRIVE_LOG_DIR'] = f'{DRIVE_ROOT}/run_logs'

# --- Sanity check: show the exact paths the trainer will use, so a
# wrong version / stale clone is impossible to miss.
print('SNC_MODEL_VERSION  =', os.environ['SNC_MODEL_VERSION'])
print('SNC_DATA_CACHE_DIR =', os.environ['SNC_DATA_CACHE_DIR'])
print('SNC_DRIVE_LOG_DIR  =', os.environ['SNC_DRIVE_LOG_DIR'])
print()
sys.path.insert(0, 'src/model_training')
import model_config as cfg
print('Will write checkpoint  :', cfg.model_path())
print('Will write class names :', cfg.classes_path())
print('Will write allow-list  :', cfg.detect_allowlist_path())
print()
# Show the last 3 git commits — confirms the clone is up to date.
subprocess.run(['git', 'log', '--oneline', '-3'], check=True)

## 2. Symlink data and saved_models to Drive

In [ ]:
drive_data = Path(DRIVE_ROOT) / 'data'
drive_models = Path(DRIVE_ROOT) / 'saved_models'
drive_data.mkdir(parents=True, exist_ok=True)
drive_models.mkdir(parents=True, exist_ok=True)

# saved_models — flat symlink to Drive (no nested symlinks needed).
local_models = REPO_DIR / 'saved_models'
if local_models.is_symlink() or local_models.exists():
    local_models.unlink() if local_models.is_symlink() else shutil.rmtree(local_models)
local_models.symlink_to(drive_models)
print(f'{local_models} -> {drive_models}')

# data — symlink each dataset subdirectory to Drive individually.
# A flat data → Drive symlink would cause cell-9 to fail:
# Google Drive does not support creating symlinks inside it (errno 95),
# and cell-9 needs to place fsd50k on local SSD, not on Drive.
raw_dir = REPO_DIR / 'data' / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)
for sub in ['archive', 'urbansound8k']:
    local = raw_dir / sub
    target = drive_data / 'raw' / sub
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        shutil.rmtree(local)
    local.symlink_to(target)
    print(f'{local} -> {target}')

## Stage 0 — Download ESC-50 (only if missing)

In [ ]:
import zipfile, urllib.request

archive_dir = REPO_DIR / 'data' / 'raw' / 'archive'
csv_path = archive_dir / 'esc50.csv'
wav_dir = archive_dir / 'audio' / 'audio'

if csv_path.exists() and wav_dir.exists() and len(list(wav_dir.glob('*.wav'))) >= 2000:
    print(f'ESC-50 already present — {len(list(wav_dir.glob("*.wav")))} WAV files.')
else:
    archive_dir.mkdir(parents=True, exist_ok=True)
    zip_path = Path('/content/esc50.zip')
    url = 'https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip'
    print('Downloading ESC-50 (~600 MB on Colab network)...')
    urllib.request.urlretrieve(url, zip_path)

    extract_dir = Path('/content/esc50_extracted')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    src_root = extract_dir / 'ESC-50-master'

    shutil.copy(src_root / 'meta' / 'esc50.csv', csv_path)
    wav_dir.mkdir(parents=True, exist_ok=True)
    print('Copying WAV files to Drive...')
    for wav in (src_root / 'audio').glob('*.wav'):
        shutil.copy(wav, wav_dir / wav.name)

    zip_path.unlink()
    shutil.rmtree(extract_dir)
    print(f'Done — {len(list(wav_dir.glob("*.wav")))} WAV files.')

assert csv_path.exists() and wav_dir.exists()

ESC-50 already present — 2000 WAV files.


In [ ]:
# Stage 0b — Download UrbanSound8K (~6 GB, only if missing).
# Adds 10 urban classes; 4 merge into ESC-50 classes, 6 are new.
import tarfile, time

us8k_dir = REPO_DIR / 'data' / 'raw' / 'urbansound8k'
us8k_csv = us8k_dir / 'metadata' / 'UrbanSound8K.csv'

if us8k_csv.exists():
    print('UrbanSound8K already present.')
else:
    tar_path = Path('/content/urbansound8k.tar.gz')
    url = 'https://zenodo.org/record/1203745/files/UrbanSound8K.tar.gz'
    print('Downloading UrbanSound8K (~6 GB on Colab network)...')

    # wget --continue resumes partial downloads; --tries=20 handles transient
    # Zenodo 504 / Gateway Timeout errors without restarting from zero.
    _wget_ok = subprocess.run(['which', 'wget'], capture_output=True).returncode == 0
    if _wget_ok:
        subprocess.run([
            'wget', '--continue', '--tries=20', '--retry-connrefused',
            '--waitretry=60', '--timeout=120', '--read-timeout=300',
            '-O', str(tar_path), url,
        ], check=True)
    else:
        # Fallback: chunked Python downloader with Range-header resume.
        existing = tar_path.stat().st_size if tar_path.exists() else 0
        headers = {'Range': f'bytes={existing}-'} if existing else {}
        req = urllib.request.Request(url, headers=headers)
        for _attempt in range(1, 21):
            try:
                with urllib.request.urlopen(req, timeout=120) as resp:
                    mode = 'ab' if existing and resp.status == 206 else 'wb'
                    with open(tar_path, mode) as f:
                        while True:
                            chunk = resp.read(1 << 20)
                            if not chunk:
                                break
                            f.write(chunk)
                break
            except Exception as exc:
                if _attempt == 20:
                    raise
                _wait = min(60, 5 * 2 ** (_attempt - 1))
                print(f'  attempt {_attempt} failed ({exc}), retrying in {_wait}s...')
                time.sleep(_wait)

    print('Extracting...')
    extract_dir = Path('/content/us8k_extract')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    with tarfile.open(tar_path) as t:
        t.extractall(extract_dir)

    # Extracted layout: us8k_extract/UrbanSound8K/{audio,metadata}
    us8k_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(extract_dir / 'UrbanSound8K'), str(us8k_dir))
    tar_path.unlink()
    shutil.rmtree(extract_dir, ignore_errors=True)
    print('UrbanSound8K ready at', us8k_dir)

assert us8k_csv.exists()

In [ ]:
# Stage 0c — FSD50K skipped.
#
# This branch trains on ESC-50 + UrbanSound8K only (~56 classes).
# Reasons: FSD50K requires 18-24 GB of downloads per Colab session,
# adds >170 classes without local-audio counterparts (phantom false
# positives at detection eval), and its long tail is pruned by the
# min_clips_per_class=40 floor anyway. The 56-class vocabulary covers
# the most common real-world sounds for selective noise-cancelling.
#
# dataset_sources.load_all_datasets() skips FSD50K automatically when
# data/raw/fsd50k/ is absent — no code change needed.
print('Stage 0c: FSD50K skipped — training on ESC-50 + UrbanSound8K only.')


## 3. Install dependencies

In [ ]:
!pip install -q librosa==0.11.0 soundfile scikit-learn pandas
import tensorflow as tf
print('TF version :', tf.__version__)
print('GPUs       :', tf.config.list_physical_devices('GPU'))

TF version : 2.20.0
GPUs       : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# === Drive logging helpers ===
# Every stage cell below runs through scripts/run_logged.py, which mirrors
# stdout+stderr into $SNC_DRIVE_LOG_DIR/runs/{timestamp}_{name}.log on Drive.
# The cell still shows live output AND the log persists if the frontend dies.
#
# Pass --background to run the command detached (cell returns immediately,
# log file fills up regardless of frontend). Useful when you expect the
# Colab tab to disconnect (long trainings, large evals).
from pathlib import Path
RUNS_DIR = Path(os.environ['SNC_DRIVE_LOG_DIR']) / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)


def list_logs(n=10):
    """Show the most recent N per-run log files on Drive."""
    files = sorted(RUNS_DIR.glob('*.log'),
                   key=lambda p: p.stat().st_mtime, reverse=True)[:n]
    for f in files:
        size = f.stat().st_size
        # Last line: '# exit: N' if finished, else the most recent stdout line.
        try:
            last = f.read_text().splitlines()[-1][:100]
        except Exception:
            last = '(unreadable)'
        print(f'{f.name:<60} {size:>10}  {last}')


def tail_log(name_substr, n=80):
    """Tail the most recent log whose filename contains ``name_substr``."""
    matches = sorted(RUNS_DIR.glob(f'*{name_substr}*.log'),
                     key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        print(f'No log matching *{name_substr}*')
        return
    print(f'=== {matches[0]} ===')
    print('\n'.join(matches[0].read_text().splitlines()[-n:]))


print('Drive runs dir :', RUNS_DIR)
print()
print('Usage:')
print('  list_logs()                — list recent log files')
print('  tail_log("train")          — show tail of latest train log')
print()
list_logs()

## 4. Decoded-dataset cache — fast session boot

Decoding ~10 000 WAV files (ESC-50 + UrbanSound8K) from Drive takes
a few minutes. The cell below mirrors the decoded result as a single
pickle between Drive and the local SSD, so the dataset is decoded
**once** ever and every later session loads it in under a minute.

* **First run:** decodes from raw and uploads the cache to Drive
  (`~3-5 min`). One time only.
* **Every later run:** copies the cached pickle from Drive to SSD
  (`~30-90 s` for one large file).

If you change CLASS_ALIASES, add a dataset, or otherwise want a fresh
decode, append `--force-rebuild` to the script call.

In [ ]:
# Cache prep — copies the decoded-dataset pickle between Drive and SSD.
# Safe to re-run: if the SSD copy is up to date it returns immediately.
os.environ['SNC_DATA_CACHE_DIR']  = '/content'
os.environ['SNC_DRIVE_CACHE_DIR'] = f'{DRIVE_ROOT}/cache'

# Show what's on Drive vs SSD before we touch anything — handy when
# debugging why training is still slow.
!python scripts/prep_data_cache.py --info

# Default mode: fetch from Drive, else decode and mirror.
!python scripts/prep_data_cache.py

## Stage 1 — Train the conditioned separator

Mixtures are generated on the fly — no dataset file. With mixed precision
+ XLA (on by default), expect roughly **20-40 min on L4/A100** or
**40-75 min on T4**, depending on `epochs` / `steps_per_epoch` and early
stopping. The best checkpoint is written to Drive as it improves.

The first lines of output confirm the speed config:
`Mixed precision: mixed_float16`, `XLA (jit_compile): True`,
`Batch size: 32`.

In [ ]:
# Train. The trainer's __main__ block applies per-version defaults:
#   v2.6:  negative_prob=0.25, detection head (BCE @ weight 0.3, dim 128)
#   v2.7:  negative_prob=0.50, detection head (focal @ weight 1.0, dim 256
#          + dropout 0.3), refreshed over-firing list from v2.6 sweep FPs
#   v2.8:  negative_prob=0.50, detection head (BCE @ weight 0.5, dim 128,
#          no dropout) — focal reverted; clean 56-class ESC+US vocab
#   v3.0:  v2.8 recipe on a curated 15-class vocabulary (keep_classes) —
#          the highest-F1 ESC-50+US8K classes only; no over-firing list
# See src/model_training/train_conditioned_separator.py for the full table.
#
# Output is mirrored to Drive at
#   $SNC_DRIVE_LOG_DIR/runs/{timestamp}_train_<version>.log
# AND a structured entry is appended to run_log.json at the end of training.
_version = os.environ['SNC_MODEL_VERSION']
!python scripts/run_logged.py train_{_version} -- \
    python src/model_training/train_conditioned_separator.py

# Verify the trained file landed at the expected path.
from pathlib import Path
import json
ckpt = cfg.model_path()
names_file = cfg.classes_path()
assert ckpt.exists(), f'Trained checkpoint missing: {ckpt}'
assert names_file.exists(), f'Class names file missing: {names_file}'
names = json.load(names_file.open())
print(f'\nCheckpoint : {ckpt} ({ckpt.stat().st_size/1e6:.1f} MB)')
print(f'Classes    : {names_file} ({len(names)} classes)')
print(f'Drive log  : {os.environ["SNC_DRIVE_LOG_DIR"]}/run_log.json')

## Stage 2 — Smoke test

In [ ]:
_version = os.environ['SNC_MODEL_VERSION']
!python scripts/run_logged.py diagnose_{_version} -- \
    python src/model_training/diagnose_model.py

## Stage 3 — Evaluate (SI-SDR)

In [ ]:
_version = os.environ['SNC_MODEL_VERSION']
!python scripts/run_logged.py sisdr_{_version} -- \
    python src/model_training/evaluate_conditioned_separator.py

## Stage 4 — Detection (precision / recall on synthetic multi-class mixtures)

In [ ]:
# Detection evaluation. v2.6 models have a detection head — the scorer
# uses the sigmoid class_presence probability instead of mask energy.
# Look for 'has_detection_head: True' in the Drive log entry.
_version = os.environ['SNC_MODEL_VERSION']
!python scripts/run_logged.py detection_{_version} -- \
    python src/model_training/evaluate_detection.py \
        --allowlist auto --write-allowlist

## Stage 4b — Detection threshold sweep (v2.6 fast diagnostic)

The default cutoffs (`floor=0.05`, `cap=0.80`, `top_k=5`) were tuned for
the v2.5-era mask-energy heuristic — scores in the 0–0.2 range. The v2.6
detection head emits sigmoid **probabilities** in `[0, 1]`, so the old
floor of 0.05 lets every class through and detection ends up at full
`top_k` every mixture.

This cell sweeps three reasonable threshold sets on the existing checkpoint
(no retraining) and writes each result to a Drive log file. Compare the
mean F1 across runs to decide whether v2.7 retraining is needed.

In [ ]:
# Three threshold sets, each logged to its own Drive file so disconnects
# don't lose anything. After all three finish, the mean F1 lines from the
# tail of each log tell you which threshold set actually works.
_version = os.environ['SNC_MODEL_VERSION']

sweeps = [
    # (label, floor, cap, top_k)
    ('moderate', 0.30, 0.90, 4),
    ('strict',   0.50, 0.95, 3),
    ('strict2',  0.70, 0.95, 2),
]
for label, floor, cap, top_k in sweeps:
    name = f'detect_sweep_{_version}_{label}'
    print(f'\n=== {label}: floor={floor} cap={cap} top_k={top_k} ===')
    !python scripts/run_logged.py {name} -- \
        python src/model_training/evaluate_detection.py \
            --allowlist auto \
            --absolute-floor {floor} \
            --relative-cap {cap} \
            --top-k {top_k}

print('\n--- All sweep logs ---')
list_logs(n=5)
print('\nTail each with:  tail_log("detect_sweep_moderate")  etc.')

## Stage 5 — Remove sounds via the web app

Once training and evaluation are done, run
`notebooks/colab_webapp.ipynb` to launch the Gradio web app. Upload
any audio or video file, tick the classes to remove, and download the
cleaned result.